In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/princelubisi/dataset2-0/LLMs.pdf
/kaggle/input/datasets/princelubisi/dataset2-0/scammer-agent.pdf
/kaggle/input/datasets/princelubisi/dataset2-0/FAQ - Declaration of Interest - final.pdf
/kaggle/input/datasets/princelubisi/dataset2-0/WEF_The_Global_Cooperation_Barometer_2024.pdf


In [3]:
!pip install -q \
langchain \
langchain-community \
langchain-text-splitters \
chromadb \
unstructured \
pypdf \
sentence-transformers

In [4]:
#import os
#os.kill(os.getpid(), 9)

In [5]:
#%pip install protobuf

In [6]:
#%pip install langchain

In [7]:
#%pip install langchain-community

In [8]:
#%pip install langchain-core

In [9]:
#%pip install langchain-text-splitters

In [10]:
#%pip install langchain-ollama

In [11]:
#%pip install chromadb

In [12]:
#%pip install unstructured

In [13]:
#%pip install unstructured[pdf]

In [14]:
#%pip install pypdf

In [15]:
#%pip install ipython

In [16]:
#%pip install "langchain==0.3.27"

In [17]:
#%pip install -U transformers peft sentence-transformers --quiet

In [18]:
#%pip install  langchain-core

In [19]:
#%pip uninstall -y langchain langchain-core langchain-community langchain-text-splitters pydantic pydantic-core


In [20]:
#%pip install "pydantic>=2.6,<3" "langchain>=0.1.20,<0.2" 


In [21]:
#%pip install "langchain-core>=0.1.52,<0.2" "langchain-community>=0.0.28,<0.1"  "langchain-text-splitters>=0.0.1,<0.1"

In [22]:
# =========================
# Imports
# =========================
import os
import warnings
warnings.filterwarnings('ignore')

from IPython.display import display, Markdown

from langchain_community.document_loaders import UnstructuredPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain.retrievers.multi_query import MultiQueryRetriever

# Hugging Face / Transformers
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline,
)
from langchain_community.llms import HuggingFacePipeline


# **Imports**

In [23]:
# =========================
# Imports
# =========================
import os
import warnings
warnings.filterwarnings('ignore')

from IPython.display import display, Markdown

# Loaders
from langchain_community.document_loaders import (
    UnstructuredPDFLoader,
    UnstructuredWordDocumentLoader
)

# Splitting
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Vector DB
from langchain_community.vectorstores import Chroma
from langchain_core.embeddings import Embeddings

# LangChain core
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain.retrievers.multi_query import MultiQueryRetriever

# Transformers / HF
import torch
from typing import List
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForCausalLM,
    pipeline
)
from langchain_community.llms import HuggingFacePipeline


# **Device**

In [24]:
# =========================
# Device
# =========================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

Device: cuda


# **Custom HF Embeddings (transformers-only)**

In [25]:
# =========================
# Custom HF Embeddings (transformers-only)
# =========================
class HFTransformersEmbeddings(Embeddings):
    """
    Minimal embeddings using HuggingFace transformers only.
    Mean-pools last_hidden_state. Works on Kaggle (no sentence-transformers).
    """
    def __init__(self, model_name: str, device: str = "cpu", normalize: bool = True):
        self.device = device
        self.normalize = normalize
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.model = AutoModel.from_pretrained(model_name).to(device)
        self.model.eval()

    @torch.no_grad()
    def _embed(self, texts: List[str]) -> List[List[float]]:
        enc = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt",
        ).to(self.device)

        outputs = self.model(**enc)
        last_hidden = outputs.last_hidden_state  # [B, T, H]
        attn_mask = enc["attention_mask"].unsqueeze(-1)  # [B, T, 1]
        masked = last_hidden * attn_mask
        sums = masked.sum(dim=1)
        counts = attn_mask.sum(dim=1).clamp(min=1)
        embs = sums / counts

        if self.normalize:
            embs = torch.nn.functional.normalize(embs, p=2, dim=1)

        return embs.cpu().tolist()

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        return self._embed(texts)

    def embed_query(self, text: str) -> List[float]:
        return self._embed([text])[0]

# **Configuration**

In [26]:
# =========================
# Configuration
# =========================
DATA_DIR = "/kaggle/input/datasets/princelubisi/dataset2-0"       # <--- change this to your folder
PERSIST_DIR = "./chroma_store_multi"        # persist your vector store
COLLECTION_NAME = "multi-doc-rag"

EMBED_MODEL_NAME = "BAAI/bge-small-en-v1.5"  # small, fast, works well on Kaggle
embedding_model = HFTransformersEmbeddings(
    model_name=EMBED_MODEL_NAME,
    device=DEVICE,
    normalize=True,
)
print(f"Custom embedding model loaded: {EMBED_MODEL_NAME}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Custom embedding model loaded: BAAI/bge-small-en-v1.5


# **Load ALL documents + metadata**

In [27]:
# =========================
# Load ALL documents + metadata
# =========================
documents = []
supported_ext = (".pdf", ".docx")
file_list = [f for f in os.listdir(DATA_DIR) if f.lower().endswith(supported_ext)]
file_list.sort()

for filename in file_list:
    file_path = os.path.join(DATA_DIR, filename)

    if filename.lower().endswith(".pdf"):
        loader = UnstructuredPDFLoader(file_path=file_path)
    elif filename.lower().endswith(".docx"):
        loader = UnstructuredWordDocumentLoader(file_path=file_path)
    else:
        continue

    try:
        docs = loader.load()
        for d in docs:
            d.metadata["source"] = filename  # keep original filename as filter key
        documents.extend(docs)
        print(f"Loaded: {filename}")
    except Exception as e:
        print(f"Skipping {filename} due to loader error: {e}")

print(f"Total base documents loaded: {len(documents)}")

Loaded: FAQ - Declaration of Interest - final.pdf
Loaded: LLMs.pdf
Loaded: WEF_The_Global_Cooperation_Barometer_2024.pdf
Loaded: scammer-agent.pdf
Total base documents loaded: 4


In [28]:
# Optional: see available sources
ALL_SOURCES = sorted({d.metadata.get("source", "") for d in documents})
print("Discovered sources:", ALL_SOURCES)

Discovered sources: ['FAQ - Declaration of Interest - final.pdf', 'LLMs.pdf', 'WEF_The_Global_Cooperation_Barometer_2024.pdf', 'scammer-agent.pdf']


# **Split into chunks**

In [29]:
# =========================
# Split into chunks
# =========================
splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=50
)
chunks = splitter.split_documents(documents)
print(f"Total chunks: {len(chunks)}")

Total chunks: 334


# **Build / Persist Vector DB**

In [30]:
# =========================
# Build / Persist Vector DB
# =========================
# If you want to rebuild from scratch each run, you can clear the folder:
# import shutil; shutil.rmtree(PERSIST_DIR, ignore_errors=True)

vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name=COLLECTION_NAME,
    persist_directory=PERSIST_DIR,
)
vector_db.persist()
print(f"Vector DB created and persisted at: {PERSIST_DIR}")

Vector DB created and persisted at: ./chroma_store_multi


# **Load Mistral‑7B‑Instruct v0**

In [31]:
# =========================
# Load Mistral‑7B‑Instruct v0.3 (quantized if possible)
# =========================
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

# Try 4-bit quantization for VRAM savings
try:
    from transformers import BitsAndBytesConfig
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map="auto",
        quantization_config=bnb_config,
    )
    print("Loaded Mistral in 4‑bit.")
except Exception as e:
    print(f"4‑bit not available ({e}). Falling back to fp16/fp32.")
    dtype = torch.float16 if DEVICE == "cuda" else torch.float32
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        dtype=dtype,
        device_map="auto",
        max_memory={0: "13GiB", "cpu": "32GiB"} if DEVICE == "cuda" else None,
    )
    print(f"Loaded Mistral in {dtype} with offloading if needed.")

# Reduce VRAM during long prompts
model.generation_config.use_cache = False

gen_pipe = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    do_sample=False,
    repetition_penalty=1.05,
    return_full_text=False,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

llm = HuggingFacePipeline(pipeline=gen_pipe)
print("Mistral‑7B‑Instruct pipeline ready.")

4‑bit not available (Using `bitsandbytes` 4-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`). Falling back to fp16/fp32.


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'pad_token_id', 'repetition_penalty', 'eos_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Loaded Mistral in torch.float16 with offloading if needed.
Mistral‑7B‑Instruct pipeline ready.


# **Multi-Query Retriever**

In [32]:
# =========================
# Multi-Query Retriever
# =========================
MULTI_QUERY_PROMPT = PromptTemplate(
    input_variables=["question"],
    template=(
        "You are an AI language model assistant. Your task is to generate 3 "
        "different versions of the given user question to retrieve relevant documents from "
        "a vector database. Provide these alternative questions separated by newlines.\n\n"
        "Original question: {question}"
    ),
)

# Base retriever across *all* documents
base_retriever = vector_db.as_retriever(search_kwargs={"k": 3})

multi_retriever = MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=llm,
    prompt=MULTI_QUERY_PROMPT
)

# **RAG Prompt**

In [33]:
# =========================
# RAG Prompt
# =========================
RAG_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are a helpful assistant. Answer the question based ONLY on the following context.
If the answer is not contained in the context, say: "I don't know based on the provided document."

Context:
{context}

Question: {question}

Answer:
"""
)

# **Token budgeted context**

In [34]:
# =========================
# Token budgeted context
# =========================
def fit_context_by_tokens(texts, tokenizer, budget_tokens=3000):
    kept, total = [], 0
    for t in texts:
        n = len(tokenizer.encode(t))
        if total + n > budget_tokens:
            break
        kept.append(t)
        total += n
    return "\n\n".join(kept)


def build_context(question, doc_name=None, budget_tokens=3000, top_k=3):
    """
    If doc_name is provided, restrict retrieval to that file via metadata filter.
    Otherwise, use multi-query retriever across all docs.
    """
    if doc_name:
        # Filter to the specific source
        retriever = vector_db.as_retriever(
            search_kwargs={"k": top_k, "filter": {"source": doc_name}}
        )
        docs = retriever.get_relevant_documents(question)
    else:
        docs = multi_retriever.get_relevant_documents(question)

    texts = [d.page_content for d in docs]
    return fit_context_by_tokens(texts, tokenizer, budget_tokens=budget_tokens)

# **Final chain**

In [35]:
# =========================
# Final chain
# =========================
def chat_with_documents(question, doc_name=None, budget_tokens=3000):
    """
    Chat over multiple documents (optionally restricted to `doc_name`).
    """
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    context = build_context(question, doc_name=doc_name, budget_tokens=budget_tokens)
    prompt = RAG_PROMPT.format(context=context, question=question)

    output = llm(prompt)
    return display(Markdown(output))

# **Questions per documents**

# **FAQ Questions**

In [36]:
# =========================
# Examples
# =========================
chat_with_documents("what is a conflict of interest ?")

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



A conflict of interest describes the situation in which personal (private) or financial/business interests of an individual may unduly influence their decisions which they are to make independently and objectively on behalf of another individual or legal entity.

In [37]:
chat_with_documents("Who must complete a Declaration of Interest?")

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Salaried Staff, Supervisors/Specialists & Managers of VWGA and its subsidiaries must complete a Declaration of Interest if a potential conflict exists.

# **Can LLM Model replace Q**

In [38]:
chat_with_documents("What role does data science play ?")

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Data science plays a pivotal role in analyzing complex datasets, such as clinical trial data and real-world data, to improve patient care and advance evidence-based medicine.

In [39]:
chat_with_documents("how do medical experts help data scientists ?")

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Medical experts help data scientists by providing them with complex datasets, such as clinical trial data and real-world data (RWD), which are crucial for improving patient care and advancing evidence-based medicine. They also collaborate closely with data scientists, sharing their deep understanding of diverse medical data types, including patient clinical data and omics data.

# **Scammer agent**

In [40]:
chat_with_documents("What is scammer agent?")

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The scammer agent is a voice-enabled AI agent designed to perform the actions necessary to conduct common scams, such as logging into bank accounts, completing a two-factor authentication process, and transferring money. The agent is not complicated, with a simple design and prompts, and it can perform the actions needed to conduct common scams autonomously.

# **Global cooperation**

In [41]:
chat_with_documents("What are the Five pillars of global cooperation and in what page and name of document i can fine them ?")

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The five pillars of global cooperation are: trade and capital, innovation and technology, climate and natural capital, health and wellness, and peace and security. You can find them on page 1 of the document titled "Global Cooperation Barometer".

In [42]:
chat_with_documents("name the five pillars of global cooperation and describe them in details ?")

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The five pillars of global cooperation, as described in the context, are:

1. Trade and Capital: This pillar focuses on the economic interdependence of nations and economies. Cooperation in this area is crucial for the smooth flow of goods, services, and capital across borders, which is essential for global economic growth and development.

2. Innovation and Technology: This pillar emphasizes the importance of collaboration in the realm of technological advancement. With the rapid pace of technological change, global cooperation is necessary to ensure that innovations are shared equitably, that they are used responsibly, and that they contribute to sustainable development.

3. Climate and Natural Capital: This pillar highlights the need for global cooperation to address climate change and protect the planet's natural resources. Cooperation is essential to implement policies that reduce greenhouse gas emissions, mitigate the effects of climate change, and conserve natural resources for future generations.

4. Health and Wellness: This pillar underscores the importance of global cooperation in promoting health and well-being. Cooperation is necessary to combat diseases, improve access to healthcare, and promote healthy lifestyles on a global scale.

5. Peace and Security

# **Test with lie**

In [43]:
chat_with_documents("Who is the King of England?")

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


I don't know based on the provided document.